In [0]:
%python

dbutils.library.restartPython()

In [0]:
%python

for nb in ["01_bronze", "02_silver", "03_gold", "04_model"]:
    print("running", nb)
    dbutils.notebook.run(nb, timeout_seconds=1800)
print("pipeline finished")

In [0]:
%python
from pyspark.sql import functions as F
import mlflow

# ---------- Gold table summary ----------
g = spark.table("lender_portfolio.gold.loan_table")
matured = g.filter("matured")
default_rate = matured.agg(F.avg("default_flag")).first()[0] * 100

print("=== Gold loan table ===")
print("total loans:", g.count())
g.groupBy("matured", "default_flag").count().orderBy("matured", "default_flag").show()
print("matured loans:", matured.count())
print("default rate (matured only):", round(default_rate, 2), "%")

# ---------- Latest model results ----------
runs = mlflow.search_runs(
    experiment_names=["/Users/alliesdl@gmail.com/credit_risk_model"],
    order_by=["start_time DESC"],
    max_results=2,
)

print("\n=== Model results (test set) ===")
for _, run in runs.iterrows():
    print(f"{run['tags.mlflow.runName']}: AUC = {run['metrics.test_auc']:.3f} | "
          f"Gini = {run['metrics.test_gini']:.3f}")